In [1]:
!pip install deep_translator

In [2]:
!pip install us

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [us]


In [3]:
#Importar las librerías necesarias
import pandas as pd
import numpy as np
from google.colab import files
import us
from deep_translator import GoogleTranslator

In [4]:
#Cargar el dataset
df_cars = pd.read_csv('/content/car_prices.csv')
df_cars.head(5)

,year,make,model,trim,body,transmission,vin,state,condition,odometer,color,interior,seller,mmr,sellingprice,saledate
0,2015,Kia,Sorento,LX,SUV,automatic,5xyktca69fg566472,ca,5.0,16639.0,white,black,kia motors america inc,20500.0,21500.0,Tue Dec 16 2014 12:30:00 GMT-0800 (PST)
1,2015,Kia,Sorento,LX,SUV,automatic,5xyktca69fg561319,ca,5.0,9393.0,white,beige,kia motors america inc,20800.0,21500.0,Tue Dec 16 2014 12:30:00 GMT-0800 (PST)
2,2014,BMW,3 Series,328i SULEV,Sedan,automatic,wba3c1c51ek116351,ca,45.0,1331.0,gray,black,financial services remarketing (lease),31900.0,30000.0,Thu Jan 15 2015 04:30:00 GMT-0800 (PST)
3,2015,Volvo,S60,T5,Sedan,automatic,yv1612tb4f1310987,ca,41.0,14282.0,white,black,volvo na rep/world omni,27500.0,27750.0,Thu Jan 29 2015 04:30:00 GMT-0800 (PST)
4,2014,BMW,6 Series Gran Coupe,650i,Sedan,automatic,wba6b2c57ed129731,ca,43.0,2641.0,gray,black,financial services remarketing (lease),66000.0,67000.0,Thu Dec 18 2014 12:30:00 GMT-0800 (PST)


In [5]:
#Comprobar el tamaño del df
df_cars.shape

(558837, 16)

In [6]:
#Analizar información básica del df
df_cars.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 558837 entries, 0 to 558836
Data columns (total 16 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   year          558837 non-null  int64  
 1   make          548536 non-null  object 
 2   model         548438 non-null  object 
 3   trim          548186 non-null  object 
 4   body          545642 non-null  object 
 5   transmission  493485 non-null  object 
 6   vin           558833 non-null  object 
 7   state         558837 non-null  object 
 8   condition     547017 non-null  float64
 9   odometer      558743 non-null  float64
 10  color         558088 non-null  object 
 11  interior      558088 non-null  object 
 12  seller        558837 non-null  object 
 13  mmr           558799 non-null  float64
 14  sellingprice  558825 non-null  float64
 15  saledate      558825 non-null  object 
dtypes: float64(4), int64(1), object(11)
memory usage: 68.2+ MB


In [7]:
#Revisar filas duplicadas
df_cars.duplicated().value_counts()

,count
False,558837


In [8]:
#Revisar valores nulos
df_cars.isnull().sum().sort_values(ascending=False)

,0
transmission,65352
body,13195
condition,11820
trim,10651
model,10399
make,10301
color,749
interior,749
odometer,94
mmr,38


In [9]:
df_cars[df_cars['make'].isna()][['make','model','trim']].head(10)

,make,model,trim
742,NaN,NaN,NaN
747,NaN,NaN,NaN
766,NaN,NaN,NaN
798,NaN,NaN,NaN
803,NaN,NaN,NaN
842,NaN,NaN,NaN
893,NaN,NaN,NaN
1514,NaN,NaN,NaN
1539,NaN,NaN,NaN
2082,NaN,NaN,NaN


In [10]:
df_cars = df_cars.dropna(subset=['make', 'model', 'trim']).reset_index(drop=True)

Tras observar que las columnas 'make', 'model' y 'trim' son nulos en las mismas filas se decide eliminar dichas filas ya que no aportan mucho valor.

In [11]:
#Eliminar la columna 'vin' ya que es un identificador único de cada vehículo y no aporta valor a la generación de insights
df_cars = df_cars.drop(columns='vin')

In [12]:
#Analisis de los valores de 'state'
df_cars['state'].unique()

array(['ca', 'tx', 'pa', 'mn', 'az', 'wi', 'tn', 'md', 'fl', 'ne', 'nj',
       'nv', 'oh', 'mi', 'ga', 'va', 'sc', 'nc', 'in', 'il', 'co', 'ut',
       'mo', 'ny', 'ma', 'pr', 'or', 'la', 'wa', 'hi', 'qc', 'ab', 'on',
       'ok', 'ms', 'nm', 'al', '3vwd17aj4fm201708', 'ns',
       '3vwd17aj2fm258506', '3vwd17aj3fm276741', '3vwd17aj2fm285365',
       '3vwd17aj0fm227318', '3vwd17aj6fm218641', '3vwd17aj7fm223475',
       '3vwd17aj5fm297123', '3vwd17aj5fm219943', '3vwd17aj9fm219766',
       '3vwd17aj3fm259017', '3vwd17aj5fm206111', '3vwd17aj5fm273601',
       '3vwd17aj5fm221322', '3vwd17aj5fm268964', '3vwd17aj6fm231972',
       '3vwd17aj7fm222388', '3vwd17aj7fm218440', '3vwd17ajxfm315938',
       '3vwd17aj7fm229552', '3vwd17aj8fm298895', '3vwd17aj4fm236636',
       '3vwd17aj5fm225953', '3vwd17aj7fm326640', '3vwd17aj8fm239622',
       '3vwd17aj2fm261566'], dtype=object)

In [13]:
eliminar = ['3vwd17aj4fm201708', '3vwd17aj2fm258506', '3vwd17aj3fm276741', '3vwd17aj2fm285365',
            '3vwd17aj0fm227318', '3vwd17aj6fm218641', '3vwd17aj7fm223475', '3vwd17aj5fm297123',
            '3vwd17aj5fm219943', '3vwd17aj9fm219766', '3vwd17aj3fm259017', '3vwd17aj5fm206111',
            '3vwd17aj5fm273601', '3vwd17aj5fm221322', '3vwd17aj5fm268964', '3vwd17aj6fm231972',
            '3vwd17aj7fm222388', '3vwd17aj7fm218440', '3vwd17ajxfm315938', '3vwd17aj7fm229552',
            '3vwd17aj8fm298895', '3vwd17aj4fm236636', '3vwd17aj5fm225953', '3vwd17aj7fm326640',
            '3vwd17aj8fm239622', '3vwd17aj2fm261566']
eliminar_indices = df_cars[df_cars['state'].isin(eliminar)].index
eliminar_indices

Index([400478, 409980, 413377, 416196, 419030, 419033, 426316, 436162, 445287,
       452932, 452947, 483156, 487598, 487601, 489628, 491949, 495718, 495725,
       510549, 518960, 518973, 518977, 519549, 538313, 540686, 544076],
      dtype='int64')

In [14]:
#Analisis de los valores de 'color'
df_cars['color'].unique()

array(['white', 'gray', 'black', 'red', 'silver', 'blue', 'brown',
       'beige', 'purple', 'burgundy', '—', 'gold', 'yellow', 'green',
       'charcoal', nan, 'orange', 'off-white', 'turquoise', 'pink',
       'lime', '4802', '9410', '1167', '2172', '14872', '12655', '15719',
       '6388', '16633', '11034', '2711', '6864', '339', '18384', '9887',
       '9837', '20379', '20627', '721', '6158', '2817', '5705', '18561',
       '2846', '9562', '5001'], dtype=object)

In [15]:
eliminar = ['4802', '9410', '1167', '2172', '14872', '12655', '15719', '6388', '16633',
            '11034', '2711', '6864', '339', '18384', '9887', '9837', '20379', '20627',
            '721', '6158', '2817', '5705', '18561', '2846', '9562', '5001']
eliminar_indices = df_cars[df_cars['color'].isin(eliminar)].index
eliminar_indices

Index([400478, 409980, 413377, 416196, 419030, 419033, 426316, 436162, 445287,
       452932, 452947, 483156, 487598, 487601, 489628, 491949, 495718, 495725,
       510549, 518960, 518973, 518977, 519549, 538313, 540686, 544076],
      dtype='int64')

In [16]:
df_cars = df_cars.drop(index=eliminar_indices).reset_index(drop=True)

Se encontraron registros con valores incongruentes en las columnas 'state' y 'color'. Además, estas inconsistencias aparecen en las mismas filas para ambas variables. Como se trata de pocos registros se eliminaron.

In [17]:
#Cambiar las abreviaturas de los estados por su nombre completo
mapeo_automatico = us.states.mapping('abbr', 'name')
mapeo_automatico = {k.lower(): v for k, v in mapeo_automatico.items()}
mapeo_automatico.update({'qc': 'Quebec', 'ab': 'Alberta', 'on': 'Ontario', 'ns': 'Nueva Escocia'})

df_cars['state'] = df_cars['state'].replace(mapeo_automatico)
display(df_cars['state'].unique())

array(['California', 'Texas', 'Pennsylvania', 'Minnesota', 'Arizona',
       'Wisconsin', 'Tennessee', 'Maryland', 'Florida', 'Nebraska',
       'New Jersey', 'Nevada', 'Ohio', 'Michigan', 'Georgia', 'Virginia',
       'South Carolina', 'North Carolina', 'Indiana', 'Illinois',
       'Colorado', 'Utah', 'Missouri', 'New York', 'Massachusetts',
       'Puerto Rico', 'Oregon', 'Louisiana', 'Washington', 'Hawaii',
       'Quebec', 'Alberta', 'Ontario', 'Oklahoma', 'Mississippi',
       'New Mexico', 'Alabama', 'Nueva Escocia'], dtype=object)

In [18]:
#Crear la columna country para identificar a que país pertenece cada estado
estados_canada = ['Quebec', 'Alberta', 'Ontario', 'Nueva Escocia']
df_cars['country'] = np.where(df_cars['state'].isin(estados_canada), 'Canada', 'Estados Unidos')

In [19]:
#Vemos que porcentaje del df son registros de Canada
df_cars['country'].value_counts(normalize=True)

,proportion
country,
Estados Unidos,0.990979
Canada,0.009021


In [20]:
eliminar_indices = df_cars[df_cars['country']=='Canada'].index
df_cars = df_cars.drop(index=eliminar_indices).reset_index(drop=True)
df_cars = df_cars.drop(columns='country')

Como los registros de Canadá representan menos del 1% del total, se borran para no meter ruido al análisis. Además, al quedarnos solo con datos de Estados Unidos, la columna 'country' deja de tener sentido y se elimina también.

In [23]:
#Revisar los valores de 'color'
df_cars['color'].unique()

array(['white', 'gray', 'black', 'red', 'silver', 'blue', 'brown',
       'beige', 'purple', 'burgundy', '—', 'gold', 'yellow', 'green',
       'charcoal', nan, 'orange', 'off-white', 'turquoise', 'pink',
       'lime'], dtype=object)

In [24]:
#Revisar los valores de 'color'
df_cars['interior'].unique()

array(['black', 'beige', 'tan', '—', 'gray', 'brown', 'burgundy', 'white',
       nan, 'silver', 'off-white', 'blue', 'red', 'yellow', 'green',
       'purple', 'orange', 'gold'], dtype=object)

In [25]:
#Traducir los valores
mapeo_colores = {
    'white': 'blanco',
    'gray': 'gris',
    'black': 'negro',
    'red': 'rojo',
    'silver': 'plata',
    'blue': 'azul',
    'brown': 'marrón',
    'purple': 'púrpura',
    'burgundy': 'borgoña',
    '—': 'desconocido',
    'gold': 'dorado',
    'yellow': 'amarillo',
    'green': 'verde',
    'charcoal': 'carbón',
    'orange': 'naranja',
    'off-white': 'blanco roto',
    'turquoise': 'turquesa',
    'pink': 'rosa',
    'lime': 'lima',
    'tan': 'marrón claro'
}
df_cars['color'] = df_cars['color'].replace(mapeo_colores)
df_cars['interior'] = df_cars['interior'].replace(mapeo_colores)

In [26]:
df_cars['transmission'] = df_cars['transmission'].replace({'automatic': 'automático'})
df_cars['transmission']

,transmission
0,automático
1,automático
2,automático
3,automático
4,automático
...,...
543113,NaN
543114,automático
543115,automático
543116,automático


In [27]:
#Cambiar el formato de la columna 'saledate'
df_cars['saledate'] = pd.to_datetime(df_cars['saledate'], errors='coerce', utc=True).dt.date
df_cars['saledate'] = df_cars['saledate'].astype('datetime64[ns]')
df_cars['saledate']

/tmp/ipykernel_1229/41568716.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_cars['saledate'] = pd.to_datetime(df_cars['saledate'], errors='coerce', utc=True).dt.date


,saledate
0,2014-12-16
1,2014-12-16
2,2015-01-14
3,2015-01-28
4,2014-12-18
...,...
543113,2015-07-09
543114,2015-07-08
543115,2015-07-08
543116,2015-07-08


In [28]:
#Se eliminan las filas con valores nulos de las siguientes columnas porque representan una cantidad pequeña
df_cars = df_cars.dropna(subset=['mmr', 'saledate', 'sellingprice', 'odometer']).reset_index(drop=True)

In [29]:
#Se completan los valores nulos de las siguientes columnas con valor 'desconocido'
df_cars['color'] = df_cars['color'].fillna('desconocido')
df_cars['interior'] = df_cars['interior'].fillna('desconocido')
df_cars['body'] = df_cars['body'].fillna('desconocido')
df_cars['transmission'] = df_cars['transmission'].fillna('desconocido')

In [30]:
#Imputar los valores nulos de la columna 'condition'
df_cars['condition'] = df_cars['condition'].fillna(df_cars.groupby('year')['condition'].transform('median'))

In [31]:
df_cars.columns

Index(['year', 'make', 'model', 'trim', 'body', 'transmission', 'state',
       'condition', 'odometer', 'color', 'interior', 'seller', 'mmr',
       'sellingprice', 'saledate'],
      dtype='object')

In [32]:
#Traducir el nombre de las columnas
df_cars.rename(columns={'year': 'año', 'make': 'marca', 'model': 'modelo', 'trim': 'version', 'body': 'carroceria', 'transmission': 'transmision',
                        'state':' estado', 'odometer':'kilometraje','condition': 'condicion', 'seller': 'vendedor',
                        'mmr': 'valor_estimado', 'sellingprice': 'precio_venta', 'saledate': 'fecha_venta'}, inplace=True)

In [33]:
#Creación de la columna 'diferencia_precio'
df_cars['diferencia_precio'] = df_cars['precio_venta'] - df_cars['valor_estimado']

In [ ]:
#Descargar el archivo final
nombre_archivo = 'car_prices_clean.csv'
df_cars.to_csv(nombre_archivo, index=False, sep=';', decimal=',', encoding='utf-8-sig')
files.download(nombre_archivo)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>